# Mi perfil de stop_times

Una fila representa una visita planificada de un viaje a una parada.
Estas consultas solo leen; no modifican Bronze ni publican Silver.
Verifiqué cada consulta mediante Genie para los snapshots actuales.
No registré una ejecución completa de este notebook ni guardé resultados en Git.
Los bloques 1 y 2 exploran metadata. Los demás muestran agregados o ejemplos limitados.
El pipeline del notebook 10 no requiere ejecutar este perfil antes de cada run.


## Bloque 1 — Esquema físico disponible

Reviso 58 campos entre Bronze stop_times y las dependencias trips, stops y frequencies Silver.
Los campos de negocio de Bronze son string. Esta consulta no demuestra la validez de sus valores.


In [ ]:
%sql
-- BLOQUE 1 — Solo metadata, sin escanear filas de negocio.
SELECT table_name, column_name, data_type, ordinal_position 
FROM workspace.information_schema.columns 
WHERE table_schema = 'gtfs_bogota' 
  AND table_name IN ('bronze_gtfs_stop_times', 'silver_gtfs_trips', 'silver_gtfs_stops', 'silver_gtfs_frequencies') 
ORDER BY table_name, ordinal_position;


## Bloque 2 — Encabezados de los dos snapshots

Reviso dos encabezados. En 2025 hay seis campos; en 2026 hay ocho.
Stop_headsign y shape_dist_traveled aparecen en 2026. No deduzco su completitud a partir del encabezado.


In [ ]:
%sql
-- BLOQUE 2 — Solo metadata, sin escanear filas de negocio.
SELECT source_feed_version, source_sha256, source_file_name, source_columns_json 
FROM workspace.gtfs_bogota.audit_gtfs_source_schema_observations 
WHERE source_file_name = 'stop_times.txt' 
  AND ((source_feed_version = '2025-10-28' AND source_sha256 = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952') 
       OR (source_feed_version = '2026-09-29' AND source_sha256 = '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')) 
ORDER BY source_feed_version 
LIMIT 10;


## Bloque 3 — Formato de horas y dominio timepoint

Compruebo horas publicadas y valores 0/1 en ambos snapshots. La consulta devuelve seis filas agregadas.
Sin efectos persistentes. Reviso el agregado antes de definir una métrica Gold.


In [ ]:
%sql
WITH snapshot_data AS (
  SELECT 
    `source_feed_version`,
    `source_sha256`,
    `timepoint`,
    `arrival_time`,
    `departure_time`
  FROM `workspace`.`gtfs_bogota`.`bronze_gtfs_stop_times`
  WHERE (`source_feed_version` = '2025-10-28' AND `source_sha256` = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952')
     OR (`source_feed_version` = '2026-09-29' AND `source_sha256` = '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
),
timepoint_domains AS (
  SELECT
    `source_feed_version`,
    `timepoint`,
    COUNT(*) AS count
  FROM snapshot_data
  GROUP BY `source_feed_version`, `timepoint`
),
time_format_checks AS (
  SELECT
    `source_feed_version`,
    `source_sha256`,
    -- Formato H+:MM:SS válido (regex básico)
    SUM(CASE WHEN `arrival_time` REGEXP '^[0-9]+:[0-5][0-9]:[0-5][0-9]$' THEN 1 ELSE 0 END) AS arrival_valid_format,
    SUM(CASE WHEN `departure_time` REGEXP '^[0-9]+:[0-5][0-9]:[0-5][0-9]$' THEN 1 ELSE 0 END) AS departure_valid_format,
    
    -- Horas > 24
    SUM(CASE WHEN CAST(SPLIT(`arrival_time`, ':')[0] AS INT) >= 24 THEN 1 ELSE 0 END) AS arrival_hours_gte_24,
    SUM(CASE WHEN CAST(SPLIT(`departure_time`, ':')[0] AS INT) >= 24 THEN 1 ELSE 0 END) AS departure_hours_gte_24,
    
    -- Minutos fuera de rango (60+)
    SUM(CASE WHEN CAST(SPLIT(`arrival_time`, ':')[1] AS INT) >= 60 THEN 1 ELSE 0 END) AS arrival_minutes_invalid,
    SUM(CASE WHEN CAST(SPLIT(`departure_time`, ':')[1] AS INT) >= 60 THEN 1 ELSE 0 END) AS departure_minutes_invalid,
    
    -- Segundos fuera de rango (60+)
    SUM(CASE WHEN CAST(SPLIT(`arrival_time`, ':')[2] AS INT) >= 60 THEN 1 ELSE 0 END) AS arrival_seconds_invalid,
    SUM(CASE WHEN CAST(SPLIT(`departure_time`, ':')[2] AS INT) >= 60 THEN 1 ELSE 0 END) AS departure_seconds_invalid,
    
    COUNT(*) AS total_rows
  FROM snapshot_data
  GROUP BY `source_feed_version`, `source_sha256`
)
SELECT 
  'timepoint_domains' AS metric_type,
  `source_feed_version`,
  NULL AS source_sha256,
  `timepoint` AS dimension_value,
  `count` AS metric_value,
  NULL AS metric_name
FROM timepoint_domains

UNION ALL

SELECT
  'time_format_validation' AS metric_type,
  `source_feed_version`,
  `source_sha256`,
  NULL AS dimension_value,
  NULL AS metric_value,
  CONCAT(
    'total:', total_rows,
    ' | arr_valid:', arrival_valid_format,
    ' | dep_valid:', departure_valid_format,
    ' | arr_h>=24:', arrival_hours_gte_24,
    ' | dep_h>=24:', departure_hours_gte_24,
    ' | arr_min_inv:', arrival_minutes_invalid,
    ' | dep_min_inv:', departure_minutes_invalid,
    ' | arr_sec_inv:', arrival_seconds_invalid,
    ' | dep_sec_inv:', departure_seconds_invalid
  ) AS metric_name
FROM time_format_checks

ORDER BY metric_type, `source_feed_version`, dimension_value


## Bloque 4 — Visitas, secuencias, distancias y referencias de ubicación

Compruebo ambos snapshots completos. El bloque 6 completa NaN/Infinity y retrocesos que incluyen llegada. No uso x!=x como evidencia de NaN.
Sin efectos persistentes. Reviso el agregado antes de definir una métrica Gold.


In [ ]:
%sql
WITH snapshot_data AS (
  SELECT 
    `source_feed_version`,
    `source_sha256`,
    `trip_id`,
    `stop_id`,
    `stop_sequence`,
    TRY_CAST(`stop_sequence` AS INT) AS stop_sequence_int,
    `arrival_time`,
    `departure_time`,
    `timepoint`,
    `stop_headsign`,
    `shape_dist_traveled`,
    -- Convertir horas H+:MM:SS a segundos BIGINT
    TRY_CAST(SPLIT(`arrival_time`, ':')[0] AS BIGINT) * 3600 + 
    TRY_CAST(SPLIT(`arrival_time`, ':')[1] AS BIGINT) * 60 + 
    TRY_CAST(SPLIT(`arrival_time`, ':')[2] AS BIGINT) AS arrival_seconds,
    TRY_CAST(SPLIT(`departure_time`, ':')[0] AS BIGINT) * 3600 + 
    TRY_CAST(SPLIT(`departure_time`, ':')[1] AS BIGINT) * 60 + 
    TRY_CAST(SPLIT(`departure_time`, ':')[2] AS BIGINT) AS departure_seconds,
    TRY_CAST(`shape_dist_traveled` AS DOUBLE) AS shape_dist_double
  FROM `workspace`.`gtfs_bogota`.`bronze_gtfs_stop_times`
  WHERE (`source_feed_version` = '2025-10-28' AND `source_sha256` = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952')
     OR (`source_feed_version` = '2026-09-29' AND `source_sha256` = '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
),
with_windows AS (
  SELECT
    `source_feed_version`,
    `source_sha256`,
    `trip_id`,
    `stop_id`,
    `stop_sequence`,
    stop_sequence_int,
    `arrival_time`,
    `departure_time`,
    arrival_seconds,
    departure_seconds,
    `timepoint`,
    `stop_headsign`,
    `shape_dist_traveled`,
    shape_dist_double,
    -- Máximo tiempo conocido previo (saltando nulos)
    MAX(GREATEST(COALESCE(arrival_seconds, 0), COALESCE(departure_seconds, 0))) 
      OVER (PARTITION BY `source_feed_version`, `source_sha256`, `trip_id` 
            ORDER BY stop_sequence_int 
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS max_prev_time_seconds,
    -- Máxima distancia previa
    MAX(shape_dist_double) 
      OVER (PARTITION BY `source_feed_version`, `source_sha256`, `trip_id` 
            ORDER BY stop_sequence_int 
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS max_prev_distance,
    -- Conteo de visitas por viaje
    COUNT(*) OVER (PARTITION BY `source_feed_version`, `source_sha256`, `trip_id`) AS trip_visit_count,
    -- Primera y última posición
    ROW_NUMBER() OVER (PARTITION BY `source_feed_version`, `source_sha256`, `trip_id` ORDER BY stop_sequence_int) AS row_num,
    COUNT(*) OVER (PARTITION BY `source_feed_version`, `source_sha256`, `trip_id`) AS total_rows_in_trip
  FROM snapshot_data
),
row_flags AS (
  SELECT
    `source_feed_version`,
    `source_sha256`,
    `trip_id`,
    `stop_id`,
    stop_sequence_int,
    arrival_seconds,
    departure_seconds,
    `timepoint`,
    `stop_headsign`,
    shape_dist_double,
    trip_visit_count,
    row_num,
    total_rows_in_trip,
    -- Validaciones de secuencia
    CASE WHEN stop_sequence_int IS NULL THEN 1 ELSE 0 END AS seq_invalid,
    CASE WHEN stop_sequence_int < 0 THEN 1 ELSE 0 END AS seq_negative,
    CASE WHEN stop_sequence_int > 2147483647 THEN 1 ELSE 0 END AS seq_overflow,
    -- Llegada > salida (como segundos)
    CASE WHEN arrival_seconds IS NOT NULL AND departure_seconds IS NOT NULL 
              AND arrival_seconds > departure_seconds THEN 1 ELSE 0 END AS arrival_after_departure_secs,
    -- Tiempo actual menor al máximo previo conocido
    CASE WHEN departure_seconds IS NOT NULL AND max_prev_time_seconds IS NOT NULL
              AND departure_seconds < max_prev_time_seconds THEN 1 ELSE 0 END AS time_less_than_prev_max,
    -- Validaciones de distancia (NaN se detecta porque x != x)
    CASE WHEN shape_dist_double IS NOT NULL 
              AND NOT (shape_dist_double = shape_dist_double) THEN 1 ELSE 0 END AS dist_is_nan,
    CASE WHEN `shape_dist_traveled` IS NOT NULL AND TRIM(`shape_dist_traveled`) != ''
              AND shape_dist_double IS NULL THEN 1 ELSE 0 END AS dist_invalid,
    CASE WHEN shape_dist_double IS NOT NULL 
              AND shape_dist_double = shape_dist_double
              AND shape_dist_double < 0 THEN 1 ELSE 0 END AS dist_negative,
    CASE WHEN shape_dist_double IS NOT NULL AND max_prev_distance IS NOT NULL 
              AND shape_dist_double = max_prev_distance THEN 1 ELSE 0 END AS dist_repeated,
    CASE WHEN shape_dist_double IS NOT NULL AND max_prev_distance IS NOT NULL 
              AND shape_dist_double < max_prev_distance THEN 1 ELSE 0 END AS dist_descending,
    -- Primera llegada ausente
    CASE WHEN row_num = 1 AND (arrival_seconds IS NULL) THEN 1 ELSE 0 END AS first_arrival_missing,
    -- Última salida ausente
    CASE WHEN row_num = total_rows_in_trip AND (departure_seconds IS NULL) THEN 1 ELSE 0 END AS last_departure_missing,
    -- Horas faltantes con timepoint=1
    CASE WHEN `timepoint` = '1' AND (arrival_seconds IS NULL OR departure_seconds IS NULL) THEN 1 ELSE 0 END AS missing_time_with_tp1,
    -- Horas pares incompletas (una presente, otra ausente)
    CASE WHEN (arrival_seconds IS NULL AND departure_seconds IS NOT NULL) 
           OR (arrival_seconds IS NOT NULL AND departure_seconds IS NULL) THEN 1 ELSE 0 END AS incomplete_time_pair,
    -- Timepoint fuera de 0/1
    CASE WHEN `timepoint` NOT IN ('0', '1') THEN 1 ELSE 0 END AS tp_out_of_range,
    -- Espacios extremos o caracteres control en campos fuente
    CASE WHEN `trip_id` != TRIM(`trip_id`) OR `trip_id` RLIKE '[\\x00-\\x1F\\x7F]' THEN 1 ELSE 0 END AS trip_id_whitespace_or_control,
    CASE WHEN `stop_id` != TRIM(`stop_id`) OR `stop_id` RLIKE '[\\x00-\\x1F\\x7F]' THEN 1 ELSE 0 END AS stop_id_whitespace_or_control,
    CASE WHEN `arrival_time` != TRIM(`arrival_time`) OR `arrival_time` RLIKE '[\\x00-\\x1F\\x7F]' THEN 1 ELSE 0 END AS arrival_time_whitespace_or_control,
    CASE WHEN `departure_time` != TRIM(`departure_time`) OR `departure_time` RLIKE '[\\x00-\\x1F\\x7F]' THEN 1 ELSE 0 END AS departure_time_whitespace_or_control,
    -- stop_headsign vacío (solo 2026)
    CASE WHEN `source_feed_version` = '2026-09-29' AND (`stop_headsign` IS NULL OR TRIM(`stop_headsign`) = '') THEN 1 ELSE 0 END AS stop_headsign_empty_2026
  FROM with_windows
),
main_aggregates AS (
  SELECT
    `source_feed_version`,
    `source_sha256`,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT STRUCT(`trip_id`, stop_sequence_int)) AS distinct_keys,
    
    -- Validaciones de secuencia
    SUM(seq_invalid) AS seq_invalid_count,
    SUM(seq_negative) AS seq_negative_count,
    SUM(seq_overflow) AS seq_overflow_count,
    
    -- Validaciones de tiempo
    SUM(arrival_after_departure_secs) AS arrival_after_departure_count,
    SUM(time_less_than_prev_max) AS time_backwards_count,
    MIN(LEAST(COALESCE(arrival_seconds, 999999999), COALESCE(departure_seconds, 999999999))) AS min_time_seconds,
    MAX(GREATEST(COALESCE(arrival_seconds, 0), COALESCE(departure_seconds, 0))) AS max_time_seconds,
    
    -- Validaciones de distancia
    SUM(dist_is_nan) AS dist_nan_count,
    SUM(dist_invalid) AS dist_invalid_count,
    SUM(dist_negative) AS dist_negative_count,
    SUM(dist_repeated) AS dist_repeated_count,
    SUM(dist_descending) AS dist_descending_count,
    
    -- Viajes de una visita
    COUNT(DISTINCT CASE WHEN trip_visit_count = 1 THEN `trip_id` END) AS trips_with_one_visit,
    
    -- Primera/última hora
    SUM(first_arrival_missing) AS first_arrival_missing_count,
    SUM(last_departure_missing) AS last_departure_missing_count,
    
    -- Horas faltantes con timepoint=1
    SUM(missing_time_with_tp1) AS missing_time_with_tp1_count,
    
    -- Horas pares incompletas
    SUM(incomplete_time_pair) AS incomplete_time_pair_count,
    
    -- Timepoint fuera de rango
    SUM(tp_out_of_range) AS tp_out_of_range_count,
    
    -- Espacios/caracteres control
    SUM(trip_id_whitespace_or_control) AS trip_id_whitespace_count,
    SUM(stop_id_whitespace_or_control) AS stop_id_whitespace_count,
    SUM(arrival_time_whitespace_or_control) AS arrival_time_whitespace_count,
    SUM(departure_time_whitespace_or_control) AS departure_time_whitespace_count,
    
    -- stop_headsign vacío 2026
    SUM(stop_headsign_empty_2026) AS stop_headsign_empty_2026_count
    
  FROM row_flags
  GROUP BY `source_feed_version`, `source_sha256`
),
location_type_usage AS (
  SELECT
    st.`source_feed_version`,
    st.`source_sha256`,
    COUNT(*) AS rows_with_location_type_check,
    SUM(CASE WHEN s.`location_type` IS NULL OR s.`location_type` != 0 THEN 1 ELSE 0 END) AS rows_with_non_zero_or_null_location_type
  FROM snapshot_data st
  LEFT JOIN `workspace`.`gtfs_bogota`.`silver_gtfs_stops` s
    ON st.`stop_id` = s.`stop_id`
    AND st.`source_feed_version` = s.`source_feed_version`
    AND st.`source_sha256` = s.`source_sha256`
  GROUP BY st.`source_feed_version`, st.`source_sha256`
),
freq_trips AS (
  SELECT DISTINCT
    `source_feed_version`,
    `source_sha256`,
    `trip_id`
  FROM `workspace`.`gtfs_bogota`.`silver_gtfs_frequencies`
  WHERE (`source_feed_version` = '2025-10-28' AND `source_sha256` = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952')
     OR (`source_feed_version` = '2026-09-29' AND `source_sha256` = '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
),
freq_first_departure AS (
  SELECT
    rf.`source_feed_version`,
    rf.`source_sha256`,
    COUNT(DISTINCT rf.`trip_id`) AS freq_trips_count,
    MIN(CASE WHEN rf.row_num = 1 THEN rf.departure_seconds END) AS min_first_departure_seconds,
    MAX(CASE WHEN rf.row_num = 1 THEN rf.departure_seconds END) AS max_first_departure_seconds,
    AVG(CASE WHEN rf.row_num = 1 THEN rf.departure_seconds END) AS avg_first_departure_seconds
  FROM row_flags rf
  INNER JOIN freq_trips ft
    ON rf.`trip_id` = ft.`trip_id`
    AND rf.`source_feed_version` = ft.`source_feed_version`
    AND rf.`source_sha256` = ft.`source_sha256`
  GROUP BY rf.`source_feed_version`, rf.`source_sha256`
)
SELECT
  ma.`source_feed_version`,
  ma.`source_sha256`,
  ma.total_rows,
  ma.distinct_keys,
  ma.seq_invalid_count,
  ma.seq_negative_count,
  ma.seq_overflow_count,
  ma.arrival_after_departure_count,
  ma.time_backwards_count,
  ma.min_time_seconds,
  ma.max_time_seconds,
  ma.dist_nan_count,
  ma.dist_invalid_count,
  ma.dist_negative_count,
  ma.dist_repeated_count,
  ma.dist_descending_count,
  ma.trips_with_one_visit,
  ma.first_arrival_missing_count,
  ma.last_departure_missing_count,
  ma.missing_time_with_tp1_count,
  ma.incomplete_time_pair_count,
  ma.tp_out_of_range_count,
  ma.trip_id_whitespace_count,
  ma.stop_id_whitespace_count,
  ma.arrival_time_whitespace_count,
  ma.departure_time_whitespace_count,
  ma.stop_headsign_empty_2026_count,
  lt.rows_with_non_zero_or_null_location_type,
  fd.freq_trips_count,
  fd.min_first_departure_seconds,
  fd.max_first_departure_seconds,
  ROUND(fd.avg_first_departure_seconds, 2) AS avg_first_departure_seconds
FROM main_aggregates ma
LEFT JOIN location_type_usage lt
  ON ma.`source_feed_version` = lt.`source_feed_version`
  AND ma.`source_sha256` = lt.`source_sha256`
LEFT JOIN freq_first_departure fd
  ON ma.`source_feed_version` = fd.`source_feed_version`
  AND ma.`source_sha256` = fd.`source_sha256`
ORDER BY ma.`source_feed_version`


## Bloque 5 — Referencias y cobertura de viajes

Compruebo referencias y viajes sin visitas. location_types describe el catálogo completo de stops; el bloque 4 mide ubicaciones usadas.
Sin efectos persistentes. Reviso el agregado antes de definir una métrica Gold.


In [ ]:
%sql
WITH snapshot_data AS (
  SELECT DISTINCT
    `source_feed_version`,
    `source_sha256`,
    `trip_id`,
    `stop_id`
  FROM `workspace`.`gtfs_bogota`.`bronze_gtfs_stop_times`
  WHERE (`source_feed_version` = '2025-10-28' AND `source_sha256` = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952')
     OR (`source_feed_version` = '2026-09-29' AND `source_sha256` = '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
),
trip_refs AS (
  SELECT
    st.`source_feed_version`,
    st.`source_sha256`,
    COUNT(DISTINCT st.`trip_id`) AS distinct_trip_ids_in_stop_times,
    COUNT(DISTINCT CASE WHEN t.`trip_id` IS NULL THEN st.`trip_id` END) AS trip_ids_missing_in_silver_trips
  FROM snapshot_data st
  LEFT JOIN `workspace`.`gtfs_bogota`.`silver_gtfs_trips` t
    ON st.`trip_id` = t.`trip_id` 
    AND st.`source_feed_version` = t.`source_feed_version`
    AND st.`source_sha256` = t.`source_sha256`
  GROUP BY st.`source_feed_version`, st.`source_sha256`
),
stop_refs AS (
  SELECT
    st.`source_feed_version`,
    st.`source_sha256`,
    COUNT(DISTINCT st.`stop_id`) AS distinct_stop_ids_in_stop_times,
    COUNT(DISTINCT CASE WHEN s.`stop_id` IS NULL THEN st.`stop_id` END) AS stop_ids_missing_in_silver_stops
  FROM snapshot_data st
  LEFT JOIN `workspace`.`gtfs_bogota`.`silver_gtfs_stops` s
    ON st.`stop_id` = s.`stop_id`
    AND st.`source_feed_version` = s.`source_feed_version`
    AND st.`source_sha256` = s.`source_sha256`
  GROUP BY st.`source_feed_version`, st.`source_sha256`
),
location_types AS (
  SELECT
    s.`source_feed_version`,
    s.`source_sha256`,
    s.`location_type`,
    COUNT(DISTINCT s.`stop_id`) AS stop_count
  FROM `workspace`.`gtfs_bogota`.`silver_gtfs_stops` s
  WHERE (s.`source_feed_version` = '2025-10-28' AND s.`source_sha256` = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952')
     OR (s.`source_feed_version` = '2026-09-29' AND s.`source_sha256` = '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
  GROUP BY s.`source_feed_version`, s.`source_sha256`, s.`location_type`
),
trips_without_visits AS (
  SELECT
    t.`source_feed_version`,
    t.`source_sha256`,
    COUNT(DISTINCT t.`trip_id`) AS total_trips_in_silver,
    COUNT(DISTINCT CASE WHEN st.`trip_id` IS NULL THEN t.`trip_id` END) AS trips_without_stop_times
  FROM `workspace`.`gtfs_bogota`.`silver_gtfs_trips` t
  LEFT JOIN snapshot_data st
    ON t.`trip_id` = st.`trip_id`
    AND t.`source_feed_version` = st.`source_feed_version`
    AND t.`source_sha256` = st.`source_sha256`
  WHERE (t.`source_feed_version` = '2025-10-28' AND t.`source_sha256` = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952')
     OR (t.`source_feed_version` = '2026-09-29' AND t.`source_sha256` = '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
  GROUP BY t.`source_feed_version`, t.`source_sha256`
),
freq_with_times AS (
  SELECT
    f.`source_feed_version`,
    f.`source_sha256`,
    COUNT(DISTINCT f.`trip_id`) AS trips_with_frequencies,
    COUNT(DISTINCT CASE WHEN st.`trip_id` IS NOT NULL THEN f.`trip_id` END) AS freq_trips_with_stop_times
  FROM `workspace`.`gtfs_bogota`.`silver_gtfs_frequencies` f
  LEFT JOIN snapshot_data st
    ON f.`trip_id` = st.`trip_id`
    AND f.`source_feed_version` = st.`source_feed_version`
    AND f.`source_sha256` = st.`source_sha256`
  WHERE (f.`source_feed_version` = '2025-10-28' AND f.`source_sha256` = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952')
     OR (f.`source_feed_version` = '2026-09-29' AND f.`source_sha256` = '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
  GROUP BY f.`source_feed_version`, f.`source_sha256`
)
SELECT
  'trip_references' AS metric_type,
  tr.`source_feed_version`,
  tr.`source_sha256`,
  NULL AS dimension_value,
  NULL AS metric_value,
  CONCAT(
    'distinct_trips:', tr.distinct_trip_ids_in_stop_times,
    ' | missing_in_silver:', tr.trip_ids_missing_in_silver_trips
  ) AS metric_name
FROM trip_refs tr

UNION ALL

SELECT
  'stop_references' AS metric_type,
  sr.`source_feed_version`,
  sr.`source_sha256`,
  NULL AS dimension_value,
  NULL AS metric_value,
  CONCAT(
    'distinct_stops:', sr.distinct_stop_ids_in_stop_times,
    ' | missing_in_silver:', sr.stop_ids_missing_in_silver_stops
  ) AS metric_name
FROM stop_refs sr

UNION ALL

SELECT
  'location_types' AS metric_type,
  lt.`source_feed_version`,
  lt.`source_sha256`,
  CAST(lt.`location_type` AS STRING) AS dimension_value,
  lt.stop_count AS metric_value,
  NULL AS metric_name
FROM location_types lt

UNION ALL

SELECT
  'trips_without_visits' AS metric_type,
  tw.`source_feed_version`,
  tw.`source_sha256`,
  NULL AS dimension_value,
  NULL AS metric_value,
  CONCAT(
    'total_trips_in_silver:', tw.total_trips_in_silver,
    ' | without_stop_times:', tw.trips_without_stop_times
  ) AS metric_name
FROM trips_without_visits tw

UNION ALL

SELECT
  'frequency_trips' AS metric_type,
  fw.`source_feed_version`,
  fw.`source_sha256`,
  NULL AS dimension_value,
  NULL AS metric_value,
  CONCAT(
    'trips_with_frequencies:', fw.trips_with_frequencies,
    ' | freq_trips_with_stop_times:', fw.freq_trips_with_stop_times
  ) AS metric_name
FROM freq_with_times fw

ORDER BY metric_type, `source_feed_version`, dimension_value


## Bloque 6 — Control final de tiempos y contenido

Uso isnan, Infinity y expresiones Unicode. Comparo el menor tiempo actual con el máximo previo conocido. Dos filas resumen ambos snapshots.
Sin efectos persistentes. Reviso el agregado antes de definir una métrica Gold.


In [ ]:
%sql
WITH base AS (
 SELECT source_feed_version, source_sha256, trip_id, stop_sequence,
   try_cast(stop_sequence AS int) AS seq,
   try_cast(split(arrival_time, ':')[0] AS bigint)*3600 + try_cast(split(arrival_time, ':')[1] AS bigint)*60 + try_cast(split(arrival_time, ':')[2] AS bigint) AS arr,
   try_cast(split(departure_time, ':')[0] AS bigint)*3600 + try_cast(split(departure_time, ':')[1] AS bigint)*60 + try_cast(split(departure_time, ':')[2] AS bigint) AS dep,
   try_cast(shape_dist_traveled AS double) AS dist,
   shape_dist_traveled,
   array(trip_id, stop_id, stop_sequence, arrival_time, departure_time, timepoint, stop_headsign, shape_dist_traveled) AS fields
 FROM workspace.gtfs_bogota.bronze_gtfs_stop_times
 WHERE (source_feed_version='2025-10-28' AND source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952')
    OR (source_feed_version='2026-09-29' AND source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
), ordered AS (
 SELECT *, max(greatest(arr,dep)) OVER(PARTITION BY source_feed_version,source_sha256,trip_id ORDER BY seq ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS prior_time
 FROM base
)
SELECT source_feed_version, source_sha256, count(*) AS rows_checked,
 count_if(least(arr,dep)<prior_time) AS time_backwards_including_arrival,
 count_if(shape_dist_traveled IS NOT NULL AND trim(shape_dist_traveled)<>'' AND (dist IS NULL OR isnan(dist) OR abs(dist)=cast('Infinity' AS double) OR dist<0)) AS invalid_distance_including_nan_inf,
 count_if(exists(fields, x -> x IS NOT NULL AND x RLIKE r'[\p{Cc}\p{Cf}]')) AS rows_with_unicode_control,
 count_if(exists(fields, x -> x IS NOT NULL AND x RLIKE r'(?U)^\s*$' AND length(x)>0)) AS rows_with_whitespace_only,
 count_if(exists(fields, x -> x IS NOT NULL AND NOT (x RLIKE r'(?U)^\s*$') AND x RLIKE r'(?U)^\s|\s$')) AS rows_with_edge_whitespace,
 count_if(NOT (stop_sequence RLIKE '^[0-9]+$') OR seq IS NULL) AS invalid_sequence_lexical_or_overflow
FROM ordered GROUP BY source_feed_version,source_sha256 ORDER BY source_feed_version


## Bloque 7 — Ejemplos de viajes con una visita

Recupero como máximo diez ejemplos de 2026. Esta muestra no explica la causa ni caracteriza todos los casos.
Sin efectos persistentes. Reviso el agregado antes de definir una métrica Gold.


In [ ]:
%sql
WITH snapshot_data AS (
  SELECT 
    `trip_id`,
    `stop_id`,
    `stop_sequence`,
    `arrival_time`,
    `departure_time`,
    `timepoint`,
    COUNT(*) OVER (PARTITION BY `trip_id`) AS trip_visit_count
  FROM `workspace`.`gtfs_bogota`.`bronze_gtfs_stop_times`
  WHERE `source_feed_version` = '2026-09-29' 
    AND `source_sha256` = '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'
)
SELECT
  `trip_id`,
  `stop_id`,
  `stop_sequence`,
  `arrival_time`,
  `departure_time`,
  `timepoint`
FROM snapshot_data
WHERE trip_visit_count = 1
ORDER BY `trip_id`
LIMIT 10


## Decisiones del contrato

Conservo las 61 visitas aisladas de 2026 con advertencia.
Registro 739 viajes sin visitas como limitación de cobertura.
No invento paradas, unidades, horarios ni causas de esas ausencias.
El contrato stop_times_v1 define las reglas del procesador compartido.
La publicación en Databricks permanece pendiente de validate y publish.
